# 00 - Data Ingestion & Validation

Polymarket dataset exploration — validate schemas, check joins, profile data quality.

**Data files**:
- `data/quant.parquet` — 1.03B trades, 35 GB (clean market data, YES perspective)
- `data/users.parquet` — 1.64B user records, 45 GB (split maker/taker, BUY direction)
- `data/markets.parquet` — 1.84M markets, 281 MB (metadata)

**Time range**: Nov 2022 – Jul 2026 (3.7 years)

**Approach**: DuckDB streaming + Polars lazy for large files, pandas only for small results

In [1]:
import duckdb
import polars as pl
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
from pathlib import Path
import json

DATA_DIR = Path('../Polymarket_data/data')
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

# DuckDB connection with optimizations
con = duckdb.connect()
con.execute("SET threads=1")
con.execute("SET enable_progress_bar=false")

print('Environment ready')

Environment ready


In [2]:
# Load metadata for all three files using PyArrow (fast metadata only)
files = ['quant.parquet', 'users.parquet', 'markets.parquet']
metadata = {}

for f in files:
    pf = pq.ParquetFile(DATA_DIR / f)
    metadata[f] = {
        'num_rows': pf.metadata.num_rows,
        'num_row_groups': pf.metadata.num_row_groups,
        'num_columns': pf.metadata.num_columns,
        'columns': pf.schema.names,
        'size_bytes': (DATA_DIR / f).stat().st_size
    }
    print(f'{f}: {pf.metadata.num_rows:,} rows, {pf.metadata.num_row_groups} row groups, {pf.metadata.num_columns} cols, {(DATA_DIR / f).stat().st_size/1e9:.2f} GB')

# Save metadata
with open(OUTPUT_DIR / 'dataset_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

quant.parquet: 1,028,646,107 rows, 1027 row groups, 13 cols, 36.68 GB
users.parquet: 1,639,420,835 rows, 13342 row groups, 14 cols, 47.69 GB
markets.parquet: 1,841,683 rows, 4 row groups, 20 cols, 0.29 GB


In [3]:
# Schema comparison via PyArrow (fast, no full scan)
print('=== SCHEMAS ===')
for f in files:
    print(f'\n--- {f} ---')
    pf = pq.ParquetFile(DATA_DIR / f)
    for col in pf.schema.names:
        print(f'  {col}')

=== SCHEMAS ===

--- quant.parquet ---
  timestamp
  block_number
  transaction_hash
  log_index
  market_id
  condition_id
  event_id
  price
  usd_amount
  token_amount
  side
  maker
  taker

--- users.parquet ---
  timestamp
  block_number
  transaction_hash
  log_index
  address
  role
  direction
  usd_amount
  token_amount
  price
  market_id
  condition_id
  event_id
  nonusdc_side

--- markets.parquet ---
  id
  question
  slug
  condition_id
  token1
  token2
  answer1
  answer2
  closed
  active
  archived
  outcome_prices
  volume
  event_id
  event_slug
  event_title
  created_at
  end_date
  updated_at
  neg_risk


In [7]:
# Quant - statistics via DuckDB streaming aggregation
print('=== QUANT STATISTICS ===')

quant_stats = con.execute(f"""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT market_id) as unique_markets,
        COUNT(DISTINCT maker) as unique_makers,
        COUNT(DISTINCT taker) as unique_takers,
        AVG(price) as avg_price,
        MIN(price) as min_price,
        MAX(price) as max_price,
        AVG(usd_amount) as avg_usd,
        SUM(usd_amount) as total_volume_usd,
        COUNT(DISTINCT side) as unique_sides,
        MIN(timestamp) as min_ts,
        MAX(timestamp) as max_ts
    FROM read_parquet('{DATA_DIR / 'quant.parquet'}')
""").df()

print(f"Rows: {quant_stats['total_rows'][0]:,}")
print(f"Unique markets: {quant_stats['unique_markets'][0]:,}")
print(f"Unique makers: {quant_stats['unique_makers'][0]:,}")
print(f"Unique takers: {quant_stats['unique_takers'][0]:,}")
print(f"Price: avg={quant_stats['avg_price'][0]:.4f}, min={quant_stats['min_price'][0]:.4f}, max={quant_stats['max_price'][0]:.4f}")
print(f"USD amount: avg={quant_stats['avg_usd'][0]:.2f}, total_volume={quant_stats['total_volume_usd'][0]:,.0f}")
print(f"Timestamp range: {quant_stats['min_ts'][0]} to {quant_stats['max_ts'][0]}")

# Side distribution
side_dist = con.execute(f"""
    SELECT side, COUNT(*) as count
    FROM read_parquet('{DATA_DIR / 'quant.parquet'}')
    GROUP BY side
""").df()
print(f'Side distribution:')
for _, row in side_dist.iterrows():
    print(f"  {row['side']}: {row['count']:,}")

=== QUANT STATISTICS ===
Rows: 1,028,646,107
Unique markets: 1,535,897
Unique makers: 1,330,291
Unique takers: 2,998,912
Price: avg=0.4388, min=0.0000, max=0.9992
USD amount: avg=36.71, total_volume=37,761,801,244
Timestamp range: 1669060169 to 1784558109
Side distribution:
  SELL: 528,459,515
  BUY: 500,186,592


In [ ]:
# Users - statistics via DuckDB streaming aggregation
print('=== USERS STATISTICS ===')

users_stats = con.execute(f"""
    SELECT 
        COUNT(*) as total_rows,
        COUNT(DISTINCT address) as unique_addresses,
        COUNT(DISTINCT market_id) as unique_markets,
        AVG(price) as avg_price,
        AVG(usd_amount) as avg_usd,
        SUM(usd_amount) as total_volume_usd,
        MIN(timestamp) as min_ts,
        MAX(timestamp) as max_ts
    FROM read_parquet('{DATA_DIR / 'users.parquet'}')
""").df()

print(f"Rows: {users_stats['total_rows'][0]:,}")
print(f"Unique addresses: {users_stats['unique_addresses'][0]:,}")
print(f"Unique markets: {users_stats['unique_markets'][0]:,}")
print(f"Price: avg={users_stats['avg_price'][0]:.4f}")
print(f"USD amount: avg={users_stats['avg_usd'][0]:.2f}, total_volume={users_stats['total_volume_usd'][0]:,.0f}")
print(f"Timestamp range: {users_stats['min_ts'][0]} to {users_stats['max_ts'][0]}")

# Role & direction distribution
role_dist = con.execute(f"""
    SELECT role, COUNT(*) as count
    FROM read_parquet('{DATA_DIR / 'users.parquet'}')
    GROUP BY role
""").df()
print(f'Role distribution:')
for _, row in role_dist.iterrows():
    print(f"  {row['role']}: {row['count']:,}")

direction_dist = con.execute(f"""
    SELECT direction, COUNT(*) as count
    FROM read_parquet('{DATA_DIR / 'users.parquet'}')
    GROUP BY direction
""").df()
print(f'Direction distribution:')
for _, row in direction_dist.iterrows():
    print(f"  {row['direction']}: {row['count']:,}")

=== USERS STATISTICS ===


In [ ]:
# Markets - full statistics (small enough to load via Polars)
print('=== MARKETS STATISTICS ===')

markets_df = pl.scan_parquet(str(DATA_DIR / 'markets.parquet')).collect()

print(f'Total markets: {len(markets_df):,}')
print(f'\nVolume:')
print(f'  Total: {markets_df["volume"].sum():,.2f}')
print(f'  Mean: {markets_df["volume"].mean():,.2f}')
print(f'  Median: {markets_df["volume"].median():,.2f}')
print(f'  Max: {markets_df["volume"].max():,.2f}')
print(f'\nStatus counts:')
print(f'  Closed: {markets_df["closed"].sum():,}')
print(f'  Active: {markets_df["active"].sum():,}')
print(f'  Neg risk: {markets_df["neg_risk"].sum():,}')
print(f'\nTop 10 markets by volume:')
top = markets_df.sort('volume', descending=True).head(10)
print(top.select(['id', 'question', 'volume', 'neg_risk']).to_pandas().to_string(index=False))

In [ ]:
# Join validation: quant.market_id -> markets.id
print('=== JOIN VALIDATION ===')

# Get all market IDs from markets (small enough)
markets_ids = set(markets_df['id'].to_list())
print(f'Markets in markets.parquet: {len(markets_ids):,}')

# Sample quant market_ids via DuckDB (streaming)
quant_markets = con.execute(f"""
    SELECT DISTINCT market_id
    FROM read_parquet('{DATA_DIR / 'quant.parquet'}')
    LIMIT 1000000
""").df()['market_id'].tolist()
quant_markets_set = set(quant_markets)
print(f'Unique markets in quant (sampled): {len(quant_markets_set):,}')

overlap = quant_markets_set & markets_ids
missing = quant_markets_set - markets_ids
print(f'Overlap: {len(overlap):,} / {len(quant_markets_set):,} = {len(overlap)/len(quant_markets_set)*100:.1f}%')
if missing:
    print(f'MISSING in markets: {len(missing)} examples: {list(missing)[:5]}')
else:
    print('✓ All sampled quant markets found in markets.parquet')

# Users join validation
users_markets = con.execute(f"""
    SELECT DISTINCT market_id
    FROM read_parquet('{DATA_DIR / 'users.parquet'}')
    LIMIT 1000000
""").df()['market_id'].tolist()
users_markets_set = set(users_markets)
overlap_u = users_markets_set & markets_ids
print(f'Users markets (sampled): {len(users_markets_set):,}')
print(f'Overlap: {len(overlap_u):,} / {len(users_markets_set):,} = {len(overlap_u)/len(users_markets_set)*100:.1f}%')

In [ ]:
# Time range validation
import datetime

print('=== TIME RANGES ===')

# Quant & Users use Unix timestamps - already fetched above
q_min_ts = quant_stats['min_ts'][0]
q_max_ts = quant_stats['max_ts'][0]
print(f'Quant: {datetime.datetime.fromtimestamp(q_min_ts)} to {datetime.datetime.fromtimestamp(q_max_ts)}')

u_min_ts = users_stats['min_ts'][0]
u_max_ts = users_stats['max_ts'][0]
print(f'Users: {datetime.datetime.fromtimestamp(u_min_ts)} to {datetime.datetime.fromtimestamp(u_max_ts)}')

# Markets use datetime
print(f'Markets created: {markets_df["created_at"].min()} to {markets_df["created_at"].max()}')
print(f'Markets end_date: {markets_df["end_date"].min()} to {markets_df["end_date"].max()}')

In [ ]:
# Data quality checks via DuckDB
print('=== DATA QUALITY CHECKS ===')

# Quant null checks (sample 1%)
quant_nulls = con.execute(f"""
    SELECT 
        COUNT(*) FILTER (WHERE transaction_hash IS NULL) as tx_hash_nulls,
        COUNT(*) FILTER (WHERE price IS NULL) as price_nulls,
        COUNT(*) FILTER (WHERE usd_amount IS NULL) as usd_nulls,
        COUNT(*) FILTER (WHERE market_id IS NULL) as market_nulls,
        COUNT(*) FILTER (WHERE maker IS NULL) as maker_nulls,
        COUNT(*) FILTER (WHERE taker IS NULL) as taker_nulls
    FROM read_parquet('{DATA_DIR / 'quant.parquet'}')
    USING SAMPLE 1% (bernoulli)
""").df()
print(f'Quant nulls (1% sample):')
print(quant_nulls.to_string(index=False))

# Users null checks (sample 1%)
users_nulls = con.execute(f"""
    SELECT 
        COUNT(*) FILTER (WHERE address IS NULL) as addr_nulls,
        COUNT(*) FILTER (WHERE price IS NULL) as price_nulls,
        COUNT(*) FILTER (WHERE usd_amount IS NULL) as usd_nulls,
        COUNT(*) FILTER (WHERE market_id IS NULL) as market_nulls
    FROM read_parquet('{DATA_DIR / 'users.parquet'}')
    USING SAMPLE 1% (bernoulli)
""").df()
print(f'\nUsers nulls (1% sample):')
print(users_nulls.to_string(index=False))

# Markets null checks (full via Polars)
print(f'\nMarkets nulls (full):')
print(markets_df.null_count().to_pandas().to_string(index=False))

# Duplicate transaction_hash check (quant sample)
dup_check = con.execute(f"""
    SELECT COUNT(*) - COUNT(DISTINCT transaction_hash) as duplicates
    FROM read_parquet('{DATA_DIR / 'quant.parquet'}')
    USING SAMPLE 1% (bernoulli)
""").df()
print(f'\nDuplicate transaction_hash in quant (1% sample): {dup_check["duplicates"][0]}')

# Price bounds check
price_bounds = con.execute(f"""
    SELECT 
        MIN(price) as min_price, MAX(price) as max_price
    FROM read_parquet('{DATA_DIR / 'quant.parquet'}')
""").df()
print(f'\nPrice bounds in quant: min={price_bounds["min_price"][0]:.4f}, max={price_bounds["max_price"][0]:.4f}')

price_bounds_u = con.execute(f"""
    SELECT 
        MIN(price) as min_price, MAX(price) as max_price
    FROM read_parquet('{DATA_DIR / 'users.parquet'}')
""").df()
print(f'Price bounds in users: min={price_bounds_u["min_price"][0]:.4f}, max={price_bounds_u["max_price"][0]:.4f}')

# USD amount bounds
usd_bounds_q = con.execute(f"""
    SELECT MIN(usd_amount) as min_usd, MAX(usd_amount) as max_usd
    FROM read_parquet('{DATA_DIR / 'quant.parquet'}')
""").df()
print(f'USD amount bounds in quant: min={usd_bounds_q["min_usd"][0]:.2f}, max={usd_bounds_q["max_usd"][0]:.2f}')

usd_bounds_u = con.execute(f"""
    SELECT MIN(usd_amount) as min_usd, MAX(usd_amount) as max_usd
    FROM read_parquet('{DATA_DIR / 'users.parquet'}')
""").df()
print(f'USD amount bounds in users: min={usd_bounds_u["min_usd"][0]:.2f}, max={usd_bounds_u["max_usd"][0]:.2f}')

In [ ]:
# Save validation summary
summary = {
    'quant': {
        'total_rows': int(quant_stats['total_rows'][0]),
        'unique_markets': int(quant_stats['unique_markets'][0]),
        'unique_makers': int(quant_stats['unique_makers'][0]),
        'unique_takers': int(quant_stats['unique_takers'][0]),
        'avg_price': float(quant_stats['avg_price'][0]),
        'min_price': float(quant_stats['min_price'][0]),
        'max_price': float(quant_stats['max_price'][0]),
        'avg_usd': float(quant_stats['avg_usd'][0]),
        'est_total_volume': float(quant_stats['total_volume_usd'][0]),
        'side_distribution': side_dist.set_index('side')['count'].to_dict(),
        'time_start': str(datetime.datetime.fromtimestamp(q_min_ts)),
        'time_end': str(datetime.datetime.fromtimestamp(q_max_ts))
    },
    'users': {
        'total_rows': int(users_stats['total_rows'][0]),
        'unique_addresses': int(users_stats['unique_addresses'][0]),
        'unique_markets': int(users_stats['unique_markets'][0]),
        'avg_price': float(users_stats['avg_price'][0]),
        'avg_usd': float(users_stats['avg_usd'][0]),
        'est_total_volume': float(users_stats['total_volume_usd'][0]),
        'role_distribution': role_dist.set_index('role')['count'].to_dict(),
        'direction_distribution': direction_dist.set_index('direction')['count'].to_dict(),
        'time_start': str(datetime.datetime.fromtimestamp(u_min_ts)),
        'time_end': str(datetime.datetime.fromtimestamp(u_max_ts))
    },
    'markets': {
        'total_rows': len(markets_df),
        'total_volume': float(markets_df['volume'].sum()),
        'avg_volume': float(markets_df['volume'].mean()),
        'closed_count': int(markets_df['closed'].sum()),
        'active_count': int(markets_df['active'].sum()),
        'neg_risk_count': int(markets_df['neg_risk'].sum()),
        'created_start': str(markets_df['created_at'].min()),
        'created_end': str(markets_df['created_at'].max())
    },
    'join_validation': {
        'quant_markets_in_markets_pct': len(overlap)/len(quant_markets_set)*100,
        'users_markets_in_markets_pct': len(overlap_u)/len(users_markets_set)*100
    }
}

with open(OUTPUT_DIR / 'validation_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print('Validation summary saved to output/validation_summary.json')